# Fraud Detection in Electricity and Gas Consumption: Data Ingestion

Milestone 0. We load the four tables of the [Zindi challenge](https://zindi.africa/competitions/fraud-detection-in-electricity-and-gas-consumption-challenge) (client and invoice data, train and test) and document their structure and how they join.

The files are not in git. Download them from Zindi and put them in `data/` (see issue "Download Zindi data").

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")

## Load the data

In [2]:
client_train = pd.read_csv(DATA_DIR / "client_train.csv")
client_test = pd.read_csv(DATA_DIR / "client_test.csv")
invoice_train = pd.read_csv(DATA_DIR / "invoice_train.csv", low_memory=False)
invoice_test = pd.read_csv(DATA_DIR / "invoice_test.csv", low_memory=False)

tables = {
    "client_train": client_train,
    "client_test": client_test,
    "invoice_train": invoice_train,
    "invoice_test": invoice_test,
}

## Shapes

In [3]:
pd.DataFrame(
    {name: df.shape for name, df in tables.items()}, index=["rows", "columns"]
).T

,rows,columns
client_train,135493,6
client_test,58069,5
invoice_train,4476749,16
invoice_test,1939730,16


The **client** tables hold one row per client (135,493 in train, 58,069 in test). The **invoice** tables hold many rows per client: about 4.5 million in train and 1.9 million in test, so around 33 invoices per client on average.

## Client table

In [4]:
client_train.head()

,disrict,client_id,client_catg,region,creation_date,target
0,60,train_Client_0,11,101,31/12/1994,0.0
1,69,train_Client_1,11,107,29/05/2002,0.0
2,62,train_Client_10,11,301,13/03/1986,0.0
3,69,train_Client_100,11,105,11/07/1996,0.0
4,62,train_Client_1000,11,303,14/10/2014,0.0


In [5]:
client_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 135493 entries, 0 to 135492
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   disrict        135493 non-null  int64  
 1   client_id      135493 non-null  str    
 2   client_catg    135493 non-null  int64  
 3   region         135493 non-null  int64  
 4   creation_date  135493 non-null  str    
 5   target         135493 non-null  float64
dtypes: float64(1), int64(3), str(2)
memory usage: 6.2 MB


Columns: `disrict` (spelled like this in the file), `client_id`, `client_catg`, `region`, `creation_date` and `target`. The test table has the same columns without `target`.

- `creation_date` is text in the format `dd/mm/yyyy`. We convert it to a date during feature engineering.
- `disrict`, `client_catg` and `region` are numeric codes, so they are **categories**, not quantities.
- There are no missing values and no duplicate rows.

In [6]:
client_train["target"].value_counts(normalize=True).rename("share")

target
0.0    0.944159
1.0    0.055841
Name: share, dtype: float64

The target is **imbalanced**: only about 5.6% of the training clients are fraudulent (`target = 1`). Accuracy would be misleading here, so we use AUC (Zindi) plus precision, recall and F1.

## Invoice table

In [7]:
invoice_train.head()

,client_id,invoice_date,tarif_type,counter_number,counter_statue,counter_code,reading_remarque,counter_coefficient,consommation_level_1,consommation_level_2,consommation_level_3,consommation_level_4,old_index,new_index,months_number,counter_type
0,train_Client_0,2014-03-24,11,1335667,0,203,8,1,82,0,0,0,14302,14384,4,ELEC
1,train_Client_0,2013-03-29,11,1335667,0,203,6,1,1200,184,0,0,12294,13678,4,ELEC
2,train_Client_0,2015-03-23,11,1335667,0,203,8,1,123,0,0,0,14624,14747,4,ELEC
3,train_Client_0,2015-07-13,11,1335667,0,207,8,1,102,0,0,0,14747,14849,4,ELEC
4,train_Client_0,2016-11-17,11,1335667,0,207,9,1,572,0,0,0,15066,15638,12,ELEC


In [8]:
invoice_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4476749 entries, 0 to 4476748
Data columns (total 16 columns):
 #   Column                Dtype
---  ------                -----
 0   client_id             str  
 1   invoice_date          str  
 2   tarif_type            int64
 3   counter_number        int64
 4   counter_statue        str  
 5   counter_code          int64
 6   reading_remarque      int64
 7   counter_coefficient   int64
 8   consommation_level_1  int64
 9   consommation_level_2  int64
 10  consommation_level_3  int64
 11  consommation_level_4  int64
 12  old_index             int64
 13  new_index             int64
 14  months_number         int64
 15  counter_type          str  
dtypes: int64(12), str(4)
memory usage: 546.5 MB


Columns: `client_id`, `invoice_date`, `tarif_type`, `counter_number`, `counter_statue`, `counter_code`, `reading_remarque`, `counter_coefficient`, `consommation_level_1` to `consommation_level_4`, `old_index`, `new_index`, `months_number` and `counter_type`.

- `invoice_date` is text (`yyyy-mm-dd`) and must be converted to a date. The dates range from 1977 to 2019.
- `counter_type` is `ELEC` or `GAZ`, so a client can have electricity and gas meters.
- No missing values. There are only a few exact duplicate rows (11 in train, 8 in test).

In [9]:
invoice_train["counter_statue"].value_counts().head(10)

counter_statue
0      4379008
1        74036
5        20639
4         2729
3          258
2           32
46          14
A           13
618         12
769          6
Name: count, dtype: int64

**Data quality finding:** `counter_statue` should take up to five values (0 to 5), but the training file also contains odd values like `A`, `46` and `618`. This makes the whole column text in the training data, while it is numeric in the test data. We have to clean it before modelling, and in the same way for train and test.

In [10]:
print("duplicate invoice rows (train):", invoice_train.duplicated().sum())
print("duplicate invoice rows (test):", invoice_test.duplicated().sum())
print("invoice dates (train):", invoice_train["invoice_date"].min(), "to", invoice_train["invoice_date"].max())

duplicate invoice rows (train): 11
duplicate invoice rows (test): 8
invoice dates (train): 1977-06-09 to 2019-12-07


## How the tables join

In [11]:
for client, invoice, split in [
    (client_train, invoice_train, "train"),
    (client_test, invoice_test, "test"),
]:
    no_client = (~invoice["client_id"].isin(client["client_id"])).sum()
    no_invoice = (~client["client_id"].isin(invoice["client_id"])).sum()
    print(f"{split}: invoices without a client: {no_client}, clients without invoices: {no_invoice}")

invoice_train.groupby("client_id").size().describe()

train: invoices without a client: 0, clients without invoices: 0
test: invoices without a client: 0, clients without invoices: 0


count    135493.000000
mean         33.040445
std          26.092229
min           1.000000
25%          10.000000
50%          30.000000
75%          50.000000
max         439.000000
dtype: float64

Both tables are linked by **`client_id`**. Every invoice belongs to a client in its client table, and every client has at least one invoice. The relationship is **one client to many invoices**, so before modelling we must aggregate the invoices to one row per client (for example count, mean, standard deviation, trends over time) and then join them to the client table.

Train and test clients are separate (`train_Client_...` and `test_Client_...`), so the test set contains clients the model has never seen.

## Summary

| Table | Rows | Key |
| --- | --- | --- |
| client_train | 135,493 | `client_id` |
| client_test | 58,069 | `client_id` |
| invoice_train | 4,476,749 | `client_id` (many per client) |
| invoice_test | 1,939,730 | `client_id` (many per client) |

Next steps (EDA): check data quality in detail, clean `counter_statue`, convert the dates and explore fraud vs. non-fraud clients.